# AI Resume Screening System
## Project Overview
This notebook performs data exploration and feature engineering for an AI-powered Resume Screening System.

*Objectives*
- Explore and understand the resume dataset
- Clean missing and inconsistent values
- Engineer meaningful text features
- Merge multiple data sources into a unified candidate profile
- Generate `resume_text` for downstream NLP tasks

*Dataset*
- People
- Experience
- Education
- Skills
- Abilities
- Job Descriptions

### Output
A production-ready master dataset where each row represents one candidate.

### Notebook Flow
1. Import Libraries
2. Load Dataset
3. Explore Dataset
4. Data Cleaning
5. Feature Engineering
6. Aggregate Candidate Information
7. Merge All Tables
8. Create Resume Text
9. Data Validation
10. Save Processed Dataset

### 1. Data Collection

The dataset contains multiple CSV files:
- Candidate information
- Education details
- Work experience
- Skills
- Abilities

We will combine these tables to create a complete candidate profile.

In [1]:
# Load Required Libraries
import pandas as pd
import numpy as np

In [2]:
# Load Raw Dataset
people=pd.read_csv("../data/raw/resumes/01_people.csv")
abilities=pd.read_csv("../data/raw/resumes/02_abilities.csv")
education=pd.read_csv("../data/raw/resumes/03_education.csv")
experience=pd.read_csv("../data/raw/resumes/04_experience.csv")
person_skills=pd.read_csv("../data/raw/resumes/05_person_skills.csv")
skills=pd.read_csv("../data/raw/resumes/06_skills.csv")

### 2. Exploratory Data Analysis (EDA)

In this step, we will do analysis of datasets and understand about:
- Dataset structure
- Missing values
- Duplicate records
- Relationship between different tables

In [3]:
#Check dataset dimensions
print("People:", people.shape)
print("Abilities:", abilities.shape)
print("Education:", education.shape)
print("Experience:", experience.shape)
print("Person Skills:", person_skills.shape)
print("Skills:", skills.shape)

People: (54933, 5)
Abilities: (1219473, 2)
Education: (75999, 5)
Experience: (265404, 6)
Person Skills: (2483376, 2)
Skills: (226760, 1)


In [4]:
# View the columns in each dataset
print("\nPeople: ",people.columns)
print("\nAbilities: ",abilities.columns)
print("\nEducation: ",education.columns)
print("\nExperience: ",experience.columns)
print("\nPerson Skills: ",person_skills.columns)
print("\nSkills: ",skills.columns)


People:  Index(['person_id', 'name', 'email', 'phone', 'linkedin'], dtype='object')

Abilities:  Index(['person_id', 'ability'], dtype='object')

Education:  Index(['person_id', 'institution', 'program', 'start_date', 'location'], dtype='object')

Experience:  Index(['person_id', 'title', 'firm', 'start_date', 'end_date', 'location'], dtype='object')

Person Skills:  Index(['person_id', 'skill'], dtype='object')

Skills:  Index(['skill'], dtype='object')


In [5]:
# Count the unique person IDs in each dataset
print("People:", people["person_id"].nunique())
print("Education:", education["person_id"].nunique())
print("Experience:", experience["person_id"].nunique())
print("Abilities:", abilities["person_id"].nunique())
print("Person Skills:", person_skills["person_id"].nunique())

People: 54933
Education: 48075
Experience: 54933
Abilities: 54930
Person Skills: 54858


In [6]:
# Inspect records for person_id = 1 across all datasets
people[people["person_id"]==1]

,person_id,name,email,phone,linkedin
0,1,Database Administrator,NaN,NaN,NaN


In [7]:
education[education["person_id"]==1]

,person_id,institution,program,start_date,location
0,1,Lead City University,Bachelor of Science,07/2013,NaN


In [8]:
experience[experience["person_id"]==1]

,person_id,title,firm,start_date,end_date,location
0,1,Database Administrator,Family Private Care LLC,04/2017,Present,"Roswell, GA"
1,1,Database Administrator,Incomm,01/2014,02/2017,"Alpharetta, GA"


In [9]:
abilities[abilities["person_id"]==1]

,person_id,ability
0,1,Installation and Building Server
1,1,Running Backups
2,1,Recovering and Restoring Models
3,1,Support various MS SQL Server
4,1,MS SQL Server 2005/2008
5,1,environments from SQL Server
6,1,/2008R2R2/2012/2014
7,1,2005 thru SQL Server 2008r2 as
8,1,administration including
9,1,well as with SQL Server 2012 on


In [10]:
person_skills[person_skills["person_id"]==1]

,person_id,skill
0,1,Database administration
1,1,Database
2,1,Ms sql server
3,1,Ms sql server 2005
4,1,Sql server
5,1,Sql server 2005
6,1,Sql server 2008
7,1,Sql server 2008 r2
8,1,Sql server 2012
9,1,Sql


### 3. Data Cleaning

Here we will slean and prepare raw data before combining different tables.

Steps performed:
- Handling missing values
- Removing unnecessary columns
- Checking duplicate records
- Standardizing text fields

In [11]:
# Check for duplicate rows in the person_skills dataset
person_skills.duplicated().sum()

np.int64(587511)

In [12]:
#Delete duplicate records from person_skills dataset
person_skills = person_skills.drop_duplicates()

In [13]:
#Verify that duplicate rows have been removed
person_skills.duplicated().sum()

np.int64(0)

In [14]:
#Check person_skills dimensions after deletion
person_skills.shape

(1895865, 2)

### 4. Data Transformation

The dataset contains multiple records for the same candidate.

Example:
A candidate can have:
- Multiple skills
- Multiple education records
- Multiple work experiences

We transform these multiple rows into a single candidate-level record.

In [15]:
# Group skills by person ID
grouped = person_skills.groupby("person_id")

In [16]:
# Check the type of the grouped object
type(grouped)

pandas.core.groupby.generic.DataFrameGroupBy

In [17]:
# Retrieve all records for person_id = 1 from the grouped data
grouped.get_group(1)

,person_id,skill
0,1,Database administration
1,1,Database
2,1,Ms sql server
3,1,Ms sql server 2005
4,1,Sql server
5,1,Sql server 2005
6,1,Sql server 2008
7,1,Sql server 2008 r2
8,1,Sql server 2012
9,1,Sql


In [18]:
# Check for missing skill values before aggregation
person_skills["skill"].isna().sum()

np.int64(6)

In [19]:
# Remove rows with missing skill values
person_skills = person_skills.dropna(subset=["skill"])

In [20]:
# Confirm there are no missing skill values
person_skills["skill"].isna().sum()

np.int64(0)

In [21]:
# Transform multiple skill records into a single row per person
skills_per_person = (
    person_skills
    .groupby("person_id")["skill"]
    .apply(lambda skills: ", ".join(pd.unique(skills)))
    .reset_index()
)

In [22]:
# Preview the aggregated skills dataset
skills_per_person.head()

,person_id,skill
0,1,"Database administration, Database, Ms sql serv..."
1,2,"sql server management studio, visual studio, s..."
2,3,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ..."
3,4,Maintain multiple database environments (Redsh...
4,5,"Scrum, Agile software development, Product bac..."


In [23]:
# Check the shape of the aggregated skills dataset
skills_per_person.shape

(54858, 2)

In [24]:
# Display the aggregated skills for the first person
print(skills_per_person.iloc[0]["skill"])

Database administration, Database, Ms sql server, Ms sql server 2005, Sql server, Sql server 2005, Sql server 2008, Sql server 2008 r2, Sql server 2012, Sql, Sql queries, Stored procedures, Clustering, Backups, T-sql, Virtualization, R2, Maintenance, Problem solving, Shipping


In [25]:
# Aggregate unique skills for each person
skills_per_person = (
    person_skills
    .groupby("person_id")["skill"]
    .apply(lambda skills: ", ".join(pd.unique(skills)))
    .reset_index()
)

In [26]:
# Preview the updated aggregated skills dataset
skills_per_person.head()

,person_id,skill
0,1,"Database administration, Database, Ms sql serv..."
1,2,"sql server management studio, visual studio, s..."
2,3,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ..."
3,4,Maintain multiple database environments (Redsh...
4,5,"Scrum, Agile software development, Product bac..."


In [27]:
experience[experience["person_id"] == 1]

,person_id,title,firm,start_date,end_date,location
0,1,Database Administrator,Family Private Care LLC,04/2017,Present,"Roswell, GA"
1,1,Database Administrator,Incomm,01/2014,02/2017,"Alpharetta, GA"


In [28]:
# Create a descriptive text for each work experience
experience["experience_text"] = (
    experience["title"]
    + " at "
    + experience["firm"]
    + " ("
    + experience["start_date"]
    + " - "
    + experience["end_date"]
    + ")"
)

In [29]:
#View "experience_text" column
experience["experience_text"]

0         Database Administrator at Family Private Care ...
1         Database Administrator at Incomm (01/2014 - 02...
2         Database Administrator at Intercontinental Reg...
3         Oracle Database Administrator at Cognizant (06...
4         Oracle Database Administrator at Convergys (06...
                                ...                        
265399    Python Developer at Hexaware Technologies Limi...
265400    Software Developer at Vision InfoTech Pvt Ltd ...
265401         MetroBikes at MetroBikes (09/2018 - Present)
265402    Python/Flask Developer at TechJini Solutions P...
265403    Python Developer at TechJini Solutions Pvt. Lt...
Name: experience_text, Length: 265404, dtype: object

In [30]:
experience[["person_id", "experience_text"]].head()

,person_id,experience_text
0,1,Database Administrator at Family Private Care ...
1,1,Database Administrator at Incomm (01/2014 - 02...
2,2,Database Administrator at Intercontinental Reg...
3,3,Oracle Database Administrator at Cognizant (06...
4,3,Oracle Database Administrator at Convergys (06...


In [31]:
# Check for missing values in the experience text column
experience["experience_text"].isna().sum()

np.int64(6738)

In [32]:
# Fill missing values in the experience details
experience["firm"] = experience["firm"].fillna("Unknown Company")
experience["start_date"] = experience["start_date"].fillna("Unknown Start")
experience["end_date"] = experience["end_date"].fillna("Present")
experience["title"] = experience["title"].fillna("Unknown Role")

In [33]:
# Recreate the Experience Text After Handling Missing Values
experience["experience_text"] = (
    experience["title"]
    + " at "
    + experience["firm"]
    + " ("
    + experience["start_date"]
    + " - "
    + experience["end_date"]
    + ")"
)

In [34]:
#Verify that the experience text column contains no missing values
experience["experience_text"].isna().sum()

np.int64(0)

In [35]:
# Aggregate work experience by person
experience_per_person=experience.groupby("person_id")["experience_text"].apply(lambda experiences: "; ".join(experiences)).reset_index()

In [36]:
#Check for missing values
education.isna().sum()

person_id          0
institution     1569
program         7761
start_date     21129
location       23256
dtype: int64

In [37]:
#Fill the NAN with missing values
education["institution"] = education["institution"].fillna("Unknown Institution")
education["program"] = education["program"].fillna("Unknown Program")
education["start_date"] = education["start_date"].fillna("Unknown Start")
education["location"] = education["location"].fillna("Unknown Location")

In [38]:
education.head(10)

,person_id,institution,program,start_date,location
0,1,Lead City University,Bachelor of Science,07/2013,Unknown Location
1,2,lagos state university,bsc in computer science,Unknown Start,"Lagos, GU"
2,3,"JNTU - Kakinada, Andhra Pradesh",Master of Computer Applications in Science and...,2013,"Kakinada, Andhra Pradesh"
3,4,University of Informatics,Bachelor in Computer Science,06/07,June 2007
4,5,Virginia Commomwealth University,Unknown Program,08/2013,"Richmond, VA"
5,6,School of Professional and Graduate Studies/Te...,Unknown Program,Unknown Start,"Overland Park, KS"
6,6,UNIVERSITY OF ALASKA,General/Business/Science Courses,Unknown Start,"Anchorage, AK"
7,7,University Of Yaounde,Biochemistry,04/08,Yaounde
8,8,Bowie State University,Bachelor's Degree in BiologyChem / Computer Sc...,Unknown Start,Unknown Location
9,9,Bridgewater State University,Management - Information Systems,Unknown Start,Present


In [39]:
# Create a descriptive text for each education record
education["education_info"]= (
    education["program"]    + " from "
    + education["institution"]
    + ", "
    + education["location"]
    + " ("
    + education["start_date"]
    + ")"
)

In [40]:
education["education_info"].head(10)

0    Bachelor of Science from Lead City University,...
1    bsc in computer science from lagos state unive...
2    Master of Computer Applications in Science and...
3    Bachelor in Computer Science from University o...
4    Unknown Program from Virginia Commomwealth Uni...
5    Unknown Program from School of Professional an...
6    General/Business/Science Courses from UNIVERSI...
7    Biochemistry from University Of Yaounde, Yaoun...
8    Bachelor's Degree in BiologyChem / Computer Sc...
9    Management - Information Systems from Bridgewa...
Name: education_info, dtype: object

In [41]:
#Check for duplicate education records per person
education["person_id"].duplicated().sum()

np.int64(27924)

In [42]:
#Aggregate all the education information in one group per person
education_grouped=education.groupby("person_id")["education_info"].apply(lambda ed_info:"; ".join(ed_info)).reset_index()

In [43]:
education_grouped.head()

,person_id,education_info
0,1,"Bachelor of Science from Lead City University,..."
1,2,bsc in computer science from lagos state unive...
2,3,Master of Computer Applications in Science and...
3,4,Bachelor in Computer Science from University o...
4,5,Unknown Program from Virginia Commomwealth Uni...


In [44]:
education_grouped.shape

(48075, 2)

In [45]:
grouped.get_group(1)

,person_id,skill
0,1,Database administration
1,1,Database
2,1,Ms sql server
3,1,Ms sql server 2005
4,1,Sql server
5,1,Sql server 2005
6,1,Sql server 2008
7,1,Sql server 2008 r2
8,1,Sql server 2012
9,1,Sql


In [46]:
abilities.head()

,person_id,ability
0,1,Installation and Building Server
1,1,Running Backups
2,1,Recovering and Restoring Models
3,1,Support various MS SQL Server
4,1,MS SQL Server 2005/2008


In [47]:
abilities.shape

(1219473, 2)

In [48]:
#Check for missing values
abilities.isna().sum()

person_id    0
ability      0
dtype: int64

In [49]:
#Aggregate abilities per person in one record
abilities_grouped = (
    abilities
    .groupby("person_id")["ability"]
    .apply(lambda abilities: "; ".join(abilities))
    .reset_index()
)


In [50]:
abilities_grouped.shape

(54930, 2)

In [51]:
abilities_grouped["person_id"].nunique()

54930

In [52]:
abilities_grouped.head()

,person_id,ability
0,1,Installation and Building Server; Running Back...
1,2,database management systems administration; de...
2,3,Over 4+ years of Experience as Architecture; E...
3,4,SQL management; PostgresSQL; Oracle; MySQL; mi...
4,5,Scrum Master; Agile software development; Prod...


In [53]:
#Confirm dimensions of all the datasets
skills_per_person.shape

(54858, 2)

In [54]:
experience_per_person.shape

(54933, 2)

In [55]:
education_grouped.shape

(48075, 2)

In [56]:
abilities_grouped.shape

(54930, 2)

In [57]:
#Confirm column names of all the dataset before merge
skills_per_person.columns

Index(['person_id', 'skill'], dtype='object')

In [58]:
experience_per_person.columns

Index(['person_id', 'experience_text'], dtype='object')

In [59]:
education_grouped.columns

Index(['person_id', 'education_info'], dtype='object')

In [60]:
abilities_grouped.columns

Index(['person_id', 'ability'], dtype='object')

### EDA and feature engineering Summary

experience_per_person : (54933, 2)

education_grouped     : (48075, 2)

skills_per_person     : (54858, 2)

abilities_grouped     : (54930, 2)

### 5. Data Integration

Combine all candidate information into a single master dataframe.

The final dataset contains:

- Candidate details
- Education
- Experience
- Skills
- Abilities

In [61]:
# Create the master dataset from the people table
master_df = people.copy()

master_df.head()

,person_id,name,email,phone,linkedin
0,1,Database Administrator,NaN,NaN,NaN
1,2,Database Administrator,NaN,NaN,NaN
2,3,Oracle Database Administrator,NaN,NaN,NaN
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN


In [62]:
master_df = master_df.merge(
    experience_per_person,
    on="person_id",
    how="left"
)

In [63]:
master_df.head()

,person_id,name,email,phone,linkedin,experience_text
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...


In [64]:
master_df = master_df.merge(
    education_grouped,
    on="person_id",
    how="left"
)
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,..."
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...


In [65]:
master_df = master_df.merge(
    skills_per_person,
    on="person_id",
    how="left"
)
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info,skill
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,...","Database administration, Database, Ms sql serv..."
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...,"sql server management studio, visual studio, s..."
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ..."
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...,Maintain multiple database environments (Redsh...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...,"Scrum, Agile software development, Product bac..."


In [66]:
master_df = master_df.merge(
    abilities_grouped,
    on="person_id",
    how="left"
)
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info,skill,ability
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,...","Database administration, Database, Ms sql serv...",Installation and Building Server; Running Back...
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...,"sql server management studio, visual studio, s...",database management systems administration; de...
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ...",Over 4+ years of Experience as Architecture; E...
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...,Maintain multiple database environments (Redsh...,SQL management; PostgresSQL; Oracle; MySQL; mi...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...,"Scrum, Agile software development, Product bac...",Scrum Master; Agile software development; Prod...


In [67]:
#Final shape of master_df after merge
master_df.shape

(54933, 9)

In [68]:
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info,skill,ability
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,...","Database administration, Database, Ms sql serv...",Installation and Building Server; Running Back...
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...,"sql server management studio, visual studio, s...",database management systems administration; de...
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ...",Over 4+ years of Experience as Architecture; E...
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...,Maintain multiple database environments (Redsh...,SQL management; PostgresSQL; Oracle; MySQL; mi...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...,"Scrum, Agile software development, Product bac...",Scrum Master; Agile software development; Prod...


In [69]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54933 entries, 0 to 54932
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   person_id        54933 non-null  int64 
 1   name             54819 non-null  object
 2   email            1593 non-null   object
 3   phone            1833 non-null   object
 4   linkedin         8538 non-null   object
 5   experience_text  54933 non-null  object
 6   education_info   48075 non-null  object
 7   skill            54858 non-null  object
 8   ability          54930 non-null  object
dtypes: int64(1), object(8)
memory usage: 3.8+ MB


In [70]:
# Check for missing values in the merged columns
master_df[
    [
        "experience_text",
        "education_info",
        "skill",
        "ability"
    ]
].isnull().sum()

experience_text       0
education_info     6858
skill                75
ability               3
dtype: int64

In [71]:
# Replace missing values with empty strings
master_df = master_df.fillna("")

### 6. Feature Engineering

Objective: Convert structured candidate information into a single textual feature that can be used for NLP-based similarity matching.

Features combined:

- Experience
- Education
- Skills
- Abilities

The final feature:
`resume_text`

In [72]:
# Create the consolidated resume text
master_df["resume_text"] = (
    master_df["name"] + " " +
    master_df["experience_text"] + " " +
    master_df["education_info"] + " " +
    master_df["skill"] + " " +
    master_df["ability"]
)

In [73]:
master_df["resume_text"].head()

0    Database Administrator Database Administrator ...
1    Database Administrator Database Administrator ...
2    Oracle Database Administrator Oracle Database ...
3    Amazon Redshift Administrator and ETL Develope...
4    Scrum Master Scrum Master Scrum Master Scrum M...
Name: resume_text, dtype: object

In [74]:
master_df[["name", "resume_text"]].head(3)

,name,resume_text
0,Database Administrator,Database Administrator Database Administrator ...
1,Database Administrator,Database Administrator Database Administrator ...
2,Oracle Database Administrator,Oracle Database Administrator Oracle Database ...


In [75]:
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info,skill,ability,resume_text
0,1,Database Administrator,,,,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,...","Database administration, Database, Ms sql serv...",Installation and Building Server; Running Back...,Database Administrator Database Administrator ...
1,2,Database Administrator,,,,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...,"sql server management studio, visual studio, s...",database management systems administration; de...,Database Administrator Database Administrator ...
2,3,Oracle Database Administrator,,,,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ...",Over 4+ years of Experience as Architecture; E...,Oracle Database Administrator Oracle Database ...
3,4,Amazon Redshift Administrator and ETL Develope...,,,,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...,Maintain multiple database environments (Redsh...,SQL management; PostgresSQL; Oracle; MySQL; mi...,Amazon Redshift Administrator and ETL Develope...
4,5,Scrum Master Scrum Master Scrum Master,,,,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...,"Scrum, Agile software development, Product bac...",Scrum Master; Agile software development; Prod...,Scrum Master Scrum Master Scrum Master Scrum M...


### NLP Text Preprocessing

In this step, we clean and normalize resume text before converting it into numerical representations.

Steps:
- Convert text to lowercase
- Remove unwanted characters
- Remove extra spaces
- Prepare text for feature extraction

In [76]:
# Convert text into lowercase
master_df["resume_text"] = master_df["resume_text"].str.lower()
master_df["resume_text"].head(3)

0    database administrator database administrator ...
1    database administrator database administrator ...
2    oracle database administrator oracle database ...
Name: resume_text, dtype: object

In [77]:
# Remove punctuation 

# Import Python's built-in string module to access punctuation characters
import string

master_df["resume_text"] = master_df["resume_text"].str.translate(
    str.maketrans("", "", string.punctuation)
)

In [78]:
print(master_df["resume_text"].iloc[0])

database administrator database administrator at family private care llc 042017  present database administrator at incomm 012014  022017 bachelor of science from lead city university unknown location 072013 database administration database ms sql server ms sql server 2005 sql server sql server 2005 sql server 2008 sql server 2008 r2 sql server 2012 sql sql queries stored procedures clustering backups tsql virtualization r2 maintenance problem solving shipping installation and building server running backups recovering and restoring models support various ms sql server ms sql server 20052008 environments from sql server 2008r2r220122014 2005 thru sql server 2008r2 as administration including well as with sql server 2012 on installation configuration upgrades capacity planning performance tuning backup and recovery familiar with virtualization and managing sql databases in a virtual environment management of users including creationalteration grant of systemdb roles and permissions on va

In [79]:
#Step 3: Remove Extra Whitespace
master_df["resume_text"] = master_df["resume_text"].str.replace(r"\s+"," ",regex=True).str.strip()
print(master_df["resume_text"].iloc[0])

database administrator database administrator at family private care llc 042017 present database administrator at incomm 012014 022017 bachelor of science from lead city university unknown location 072013 database administration database ms sql server ms sql server 2005 sql server sql server 2005 sql server 2008 sql server 2008 r2 sql server 2012 sql sql queries stored procedures clustering backups tsql virtualization r2 maintenance problem solving shipping installation and building server running backups recovering and restoring models support various ms sql server ms sql server 20052008 environments from sql server 2008r2r220122014 2005 thru sql server 2008r2 as administration including well as with sql server 2012 on installation configuration upgrades capacity planning performance tuning backup and recovery familiar with virtualization and managing sql databases in a virtual environment management of users including creationalteration grant of systemdb roles and permissions on vari

In [80]:
# Step 4: Tokenization
#Import nltk to perform NLP preprocessing tasks
import nltk

In [81]:
#Download the tokenizer
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/swaritshrivastava/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/swaritshrivastava/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [82]:
# Import the word tokenizer
from nltk.tokenize import word_tokenize

# Convert each resume into a list of individual word tokens
master_df["tokens"] = master_df["resume_text"].apply(word_tokenize)

In [83]:
master_df[["resume_text","tokens"]].head()

,resume_text,tokens
0,database administrator database administrator ...,"[database, administrator, database, administra..."
1,database administrator database administrator ...,"[database, administrator, database, administra..."
2,oracle database administrator oracle database ...,"[oracle, database, administrator, oracle, data..."
3,amazon redshift administrator and etl develope...,"[amazon, redshift, administrator, and, etl, de..."
4,scrum master scrum master scrum master scrum m...,"[scrum, master, scrum, master, scrum, master, ..."


In [84]:
#Step 5: Remove Stopwords

# Download the English stopword corpus 
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/swaritshrivastava/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [85]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

In [86]:
master_df["tokens"]=master_df["tokens"].apply(
    lambda tokens: [word for word in tokens if word not in stop_words])

In [87]:
# Step 6: Lemmatization
# Download the WordNet lexical database
nltk.download("wordnet")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/swaritshrivastava/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [88]:
# Download the multilingual WordNet package
nltk.download("omw-1.4")

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/swaritshrivastava/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [89]:
from nltk.stem import WordNetLemmatizer

In [90]:
# Create an instance of the WordNet lemmatizer
lemmatizer =WordNetLemmatizer()

In [91]:
# Convert every token to its base (dictionary) form
master_df["tokens"] = master_df["tokens"].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)
master_df["tokens"]

0        [database, administrator, database, administra...
1        [database, administrator, database, administra...
2        [oracle, database, administrator, oracle, data...
3        [amazon, redshift, administrator, etl, develop...
4        [scrum, master, scrum, master, scrum, master, ...
                               ...                        
54928    [lead, python, developer, lead, python, develo...
54929    [full, stack, python, developer, full, stack, ...
54930    [eli, lilly, sr, python, developer, eli, lilly...
54931    [python, developer, python, developer, intuit,...
54932    [job, seeker, metrobikes, metrobikes, 092018, ...
Name: tokens, Length: 54933, dtype: object

In [92]:
#Step 7: Convert Tokens Back to Text
# Join the processed tokens into a single cleaned text string
master_df["cleaned_resume_text"] = master_df["tokens"].apply(
    lambda tokens: " ".join(tokens)
)


In [93]:
master_df[
    ["resume_text", "tokens", "cleaned_resume_text"]
].head(2)

,resume_text,tokens,cleaned_resume_text
0,database administrator database administrator ...,"[database, administrator, database, administra...",database administrator database administrator ...
1,database administrator database administrator ...,"[database, administrator, database, administra...",database administrator database administrator ...


In [94]:
# Step 8: Compare Original and Cleaned Resume Text
comparison_df = master_df[
    ["person_id", "resume_text", "cleaned_resume_text"]
].copy()

comparison_df.head(5)

,person_id,resume_text,cleaned_resume_text
0,1,database administrator database administrator ...,database administrator database administrator ...
1,2,database administrator database administrator ...,database administrator database administrator ...
2,3,oracle database administrator oracle database ...,oracle database administrator oracle database ...
3,4,amazon redshift administrator and etl develope...,amazon redshift administrator etl developer bu...
4,5,scrum master scrum master scrum master scrum m...,scrum master scrum master scrum master scrum m...


In [95]:
sample_index = 0

print("========== Original Resume ==========\n")
print(master_df.loc[sample_index, "resume_text"])

print("\n========== Cleaned Resume ==========\n")
print(master_df.loc[sample_index, "cleaned_resume_text"])

========== Original Resume ==========

database administrator database administrator at family private care llc 042017 present database administrator at incomm 012014 022017 bachelor of science from lead city university unknown location 072013 database administration database ms sql server ms sql server 2005 sql server sql server 2005 sql server 2008 sql server 2008 r2 sql server 2012 sql sql queries stored procedures clustering backups tsql virtualization r2 maintenance problem solving shipping installation and building server running backups recovering and restoring models support various ms sql server ms sql server 20052008 environments from sql server 2008r2r220122014 2005 thru sql server 2008r2 as administration including well as with sql server 2012 on installation configuration upgrades capacity planning performance tuning backup and recovery familiar with virtualization and managing sql databases in a virtual environment management of users including creationalteration grant of

In [96]:
# Step 9: Save the Cleaned Dataset
master_df.to_csv(
    "../data/processed/cleaned_resume_dataset.csv",
    index=False
)

In [97]:
# Verify that the cleaned dataset has been saved successfully
import os

file_path = "../data/processed/cleaned_resume_dataset.csv"

if os.path.exists(file_path):
    print("Cleaned dataset saved successfully!")
    print(f"Location: {file_path}")
else:
    print("File not found. Please check the path.")

Cleaned dataset saved successfully!
Location: ../data/processed/cleaned_resume_dataset.csv


### Feature Engineering

Objective: 
Transform cleaned resume text into numerical feature vectors using TF-IDF so that resumes and job descriptions can be compared mathematically for similarity-based candidate matching.

Techniques Covered
- Bag of Words (BoW)
- TF-IDF (Term Frequency–Inverse Document Frequency)
- TF-IDF Feature Matrix
- Similarity-Based Feature Representation

In [98]:
# Import the TF-IDF Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [99]:
# Create the TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

In [100]:
# Convert cleaned resume text into TF-IDF feature vectors
X = tfidf.fit_transform(master_df["cleaned_resume_text"])

In [101]:
# Display the shape of the TF-IDF feature matrix
print(X.shape)

(54933, 5000)


In [102]:
# Check the type of the generated feature matrix
print(type(X))

<class 'scipy.sparse._csr.csr_matrix'>


In [103]:
# Display the sparse TF-IDF matrix
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6999813 stored elements and shape (54933, 5000)>

In [104]:
# Exploring the TF-IDF Features
# Retrieve all feature names learned by the TF-IDF vectorizer
feature_names = tfidf.get_feature_names_out()

In [105]:
feature_names[:40]

array(['0107', '0108', '0109', '0110', '0111', '0112', '0113', '0114',
       '0115', '0116', '0117', '0118', '0118 present', '0119',
       '0119 present', '012004', '012005', '012006', '012007', '012008',
       '012009', '012010', '012011', '012012', '012013', '012014',
       '012015', '012016', '012016 present', '012017', '012017 present',
       '012018', '012018 present', '012019', '012019 present', '0211',
       '0212', '0213', '0214', '0215'], dtype=object)

### Feature Refinement

Observation:
While inspecting the generated TF-IDF vocabulary, several standalone numeric tokens originating from dates and year information (for example, `0107`, `0118`, `2022`) were identified as features. These tokens do not contribute meaningful information for resume-job matching and unnecessarily increase the vocabulary.

Improvement:
To improve feature quality, standalone numeric tokens were removed from the cleaned resume text while preserving meaningful technical identifiers such as **Oracle 11g**, **Oracle 12c**, and **SQL Server 2008 R2**. The TF-IDF vectors were then regenerated using the refined text.

Outcome:
The regenerated TF-IDF vocabulary contains more meaningful technical terms, resulting in cleaner feature representations for similarity-based resume matching.

In [106]:
#Remove Numeric Tokens
master_df["cleaned_resume_text"] = (
    master_df["cleaned_resume_text"]
    .str.replace(r"\b\d+\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [107]:
# Recreate the TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

In [108]:
# Generate the TF-IDF feature matrix using the refined text
#X = resume_tfidf_matrix
#y = target_labels
X = tfidf.fit_transform(
    master_df["cleaned_resume_text"]
)

In [109]:
# Retrieve the updated feature names
feature_names = tfidf.get_feature_names_out()

In [110]:
feature_names[:40]

array(['10g', '10g 11g', '10g oracle', '10g11g', '11g', '11g 12c',
       '11g oracle', '12c', '2008r2', '24x7', '2x', '3d', '3rd',
       '3rd party', '3x', '4x', '80053a', '8i', '9i', 'aa', 'ab',
       'ab testing', 'ability', 'ability work', 'able', 'academic',
       'academy', 'accenture', 'acceptance', 'acceptance testing',
       'access', 'access control', 'access database', 'access management',
       'access object', 'access point', 'accessibility', 'accordance',
       'according', 'account'], dtype=object)

### Generating a TF-IDF Vector for a Job Description

Objective
To compare resumes with a job description, both must be represented in the same numerical feature space. In this step, the trained TF-IDF vectorizer is used to convert a job description into a TF-IDF feature vector.

The TF-IDF vectorizer has already learned the vocabulary from the resume dataset. Instead of learning a new vocabulary, the same vectorizer is used to transform job descriptions into the existing feature space.

Using the same vocabulary ensures that resumes and job descriptions can be compared directly using similarity measures such as Cosine Similarity.

Outcome:
- Converts a job description into a TF-IDF vector.
- Maintains the same feature space as the resume vectors.
- Prepares the data for similarity-based resume matching.

In [111]:
#Load the job descriptions dataset
job_df = pd.read_csv("../data/raw/job_descriptions/job_descriptions.csv")

In [112]:
# Display the first five records
job_df.head()

,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [113]:
# Display column names
print(job_df.columns)

Index(['Unnamed: 0', 'Job Title', 'Job Description'], dtype='object')


In [114]:
# Remove the unnecessary index column
job_df = job_df.drop(columns=["Unnamed: 0"])

In [115]:
# Display dataset shape
print(job_df.shape)

(2277, 2)


In [116]:
#Handle missing value(if any): Check missing values
job_df.isnull().sum()

Job Title          0
Job Description    0
dtype: int64

In [117]:
# Convert job descriptions into TF-IDF vectors using the existing vectorizer
job_tfidf_matrix = tfidf.transform(job_df["Job Description"])

In [118]:
# Display the shape of the job TF-IDF matrix
print(job_tfidf_matrix.shape)

(2277, 5000)


In [119]:
# Check the type of the generated job description TF-IDF matrix
print(type(job_tfidf_matrix))

<class 'scipy.sparse._csr.csr_matrix'>


In [120]:
job_df

,Job Title,Job Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,Full Stack Developer,job responsibility full stack engineer – react...
...,...,...
2272,Backend Developer,Job Summary\nPublished on : 26 days ago\nVacan...
2273,Full Stack Developer,business entity cisco umbrella focus cloud-bas...
2274,Network Administrator,Urgently reqd in a college in Mohali\nNetwork ...
2275,Machine Learning,Key Responsibilities: Team leads for small or ...


### Resume Matching using Cosine Similarity

After converting resumes and job descriptions into TF-IDF vectors, the next step is to measure how similar they are.

Cosine Similarity calculates the angle between two vectors and provides a similarity score between 0 and 1, where higher scores indicate a better match.

This technique forms the core matching algorithm of the Resume Screening System.

In [121]:
# Import cosine similarity function
from sklearn.metrics.pairwise import cosine_similarity

In [122]:
# Calculate similarity scores between every resume and every job description
similarity_matrix = cosine_similarity(X, job_tfidf_matrix)

In [123]:
# Display the dimensions of the similarity matrix
print(similarity_matrix.shape)

(54933, 2277)


In [124]:
# Display similarity scores for the first five job descriptions
print(similarity_matrix[0][:5])

[0.02061718 0.05753149 0.03768483 0.02648955 0.07853234]


### Retrieving the Top Matching Jobs

The similarity matrix contains a score for every resume-job pair. In this step, we identify the job descriptions with the highest similarity scores for a given resume.

These top-ranked jobs represent the most relevant recommendations generated by the Resume Screening System.

In [125]:
# Get indices of the top 10 matching jobs for the first resume
top_10_indices = np.argsort(similarity_matrix[0])[::-1][:10]

# Display the indices
print(top_10_indices)

[1938 1824 1317 2007 1010  284 1775 1348 1963  728]


In [126]:
# Display the top 10 recommended job titles and their similarity scores
for index in top_10_indices:
    print(f"Job Title: {job_df.iloc[index]['Job Title']}")
    print(f"Similarity Score: {similarity_matrix[0][index]:.4f}")
    print("-" * 50)

Job Title: Database Administrator
Similarity Score: 0.4686
--------------------------------------------------
Job Title: Database Administrator
Similarity Score: 0.3850
--------------------------------------------------
Job Title: Database Administrator
Similarity Score: 0.3495
--------------------------------------------------
Job Title: Database Administrator
Similarity Score: 0.3071
--------------------------------------------------
Job Title: Database Administrator
Similarity Score: 0.3065
--------------------------------------------------
Job Title: Database Administrator
Similarity Score: 0.2958
--------------------------------------------------
Job Title: Database Administrator
Similarity Score: 0.2866
--------------------------------------------------
Job Title: Database Administrator
Similarity Score: 0.2717
--------------------------------------------------
Job Title: Database Administrator
Similarity Score: 0.2673
--------------------------------------------------
Job Title:

### Creating a Structured Recommendation Table

Printing recommendations is useful for debugging, but a structured DataFrame is easier to analyze, export, and integrate into APIs or web applications.

In this step, we convert the recommended job titles and their similarity scores into a pandas DataFrame.

In [127]:
# Create an empty list to store the recommendations
recommendations = []

# Store the top 10 recommended jobs and their similarity scores
for index in top_10_indices:
    recommendations.append({
        "Job Title": job_df.iloc[index]["Job Title"],
        "Similarity Score": round(similarity_matrix[0][index], 4)
    })

In [128]:
# Convert the recommendations list into a DataFrame
recommendation_df = pd.DataFrame(recommendations)

# Display the recommendation table
recommendation_df

,Job Title,Similarity Score
0,Database Administrator,0.4686
1,Database Administrator,0.3850
2,Database Administrator,0.3495
3,Database Administrator,0.3071
4,Database Administrator,0.3065
5,Database Administrator,0.2958
6,Database Administrator,0.2866
7,Database Administrator,0.2717
8,Database Administrator,0.2673
9,Database Administrator,0.2549


**Result**: The recommendation table presents the top matching job descriptions for the selected resume.

Retrieving the Top Matching Jobs After computing the cosine similarity matrix, we rank all job descriptions based on their similarity scores for a selected resume. The top-ranked jobs represent the most relevant recommendations according to the resume's skills, experience, and education. This demonstrates the core recommendation engine of the AI Resume Screening System.

In [129]:
for index in top_10_indices:
    print("=" * 100)
    print("Job Title:", job_df.iloc[index]["Job Title"])
    print()
    print(job_df.iloc[index]["Job Description"][:40])

Job Title: Database Administrator

kirkland & elli kirkland & elli llp pree
Job Title: Database Administrator

summary/objective database administrator
Job Title: Database Administrator

database administrator mi datab01213 app
Job Title: Database Administrator

turn 5 one largest fastest growing e-com
Job Title: Database Administrator

overview database administrator ii insta
Job Title: Database Administrator

position summary sql server database adm
Job Title: Database Administrator

provide intermediate level database admi
Job Title: Database Administrator

hpd seeking sql database administrator p
Job Title: Database Administrator

looking sql server dba / database engine
Job Title: Database Administrator

apu seeking employee desire engage thriv


### Testing Recommendations on a Random Resume

To demonstrate that the recommendation system works for any resume in the dataset, we randomly select a resume and retrieve its top matching job recommendations.

In [130]:
import random

# Randomly select a resume
resume_index = random.randint(0, len(master_df) - 1)

print("Selected Resume Index:", resume_index)
print("Resume Title:", master_df.iloc[resume_index]["name"])

# Find top 10 matching jobs
top_10_indices = similarity_matrix[resume_index].argsort()[-10:][::-1]

print('-'*30)
print("Top 10 Recommended Jobs:")
print('-'*30)

for idx in top_10_indices:
    print(job_df.iloc[idx]["Job Title"])

Selected Resume Index: 2787
Resume Title: Oracle Database Administrator
------------------------------
Top 10 Recommended Jobs:
------------------------------
Database Administrator
Database Administrator
Database Administrator
Database Administrator
Database Administrator
Database Administrator
Database Administrator
Database Administrator
Database Administrator
Database Administrator


### Evaluating the Recommendation System
Since this project uses an unsupervised recommendation approach (TF-IDF + Cosine Similarity), there is no ground truth label available. Therefore, evaluation is performed by manually inspecting the recommended job titles and similarity scores for multiple resumes.

This qualitative evaluation helps verify that resumes are matched with semantically similar job descriptions.

In [131]:
# Test recommendations for multiple resumes
num_tests = 3

for i in range(num_tests):

    resume_index = random.randint(0, len(master_df) - 1)

    print("=" * 100)
    print(f"Test Case {i+1}")
    print(f"Resume Index : {resume_index}")
    print(f"Resume Title : {master_df.iloc[resume_index]['name']}")
    print()

    top_5 = similarity_matrix[resume_index].argsort()[-5:][::-1]

    for idx in top_5:
        print(
            f"{job_df.iloc[idx]['Job Title']} "
            f"(Similarity: {similarity_matrix[resume_index][idx]:.4f})"
        )

    print()

Test Case 1
Resume Index : 6302
Resume Title : Project Management

Database Administrator (Similarity: 0.1816)
Database Administrator (Similarity: 0.1696)
Database Administrator (Similarity: 0.1695)
Database Administrator (Similarity: 0.1691)
Network Administrator (Similarity: 0.1448)

Test Case 2
Resume Index : 4138
Resume Title : Lavu Albuquerque

Full Stack Developer (Similarity: 0.2445)
iOS Developer (Similarity: 0.1767)
Backend Developer (Similarity: 0.1767)
PHP Developer (Similarity: 0.1744)
Software Engineer (Similarity: 0.1726)

Test Case 3
Resume Index : 31274
Resume Title : Sr. Java Developer

Java Developer (Similarity: 0.4251)
Java Developer (Similarity: 0.3927)
Java Developer (Similarity: 0.3882)
Full Stack Developer (Similarity: 0.3674)
Full Stack Developer (Similarity: 0.3484)



In [132]:
# Similarity Score Analysis:

print("Maximum Similarity :", np.max(similarity_matrix))
print("Minimum Similarity :", np.min(similarity_matrix))
print("Average Similarity :", np.mean(similarity_matrix))

Maximum Similarity : 0.6901805381173256
Minimum Similarity : 0.0
Average Similarity : 0.044067680716034445


### Similarity Score Interpretation

The similarity score analysis provides an overview of how well resumes match the available job descriptions.

### Results

- **Maximum Similarity:** 0.6902
- **Minimum Similarity:** 0.0000
- **Average Similarity:** 0.0441

### Interpretation

- The highest similarity score of **0.6902** indicates that at least one resume has a strong textual match with a job description.
- A minimum score of **0.0000** shows that some resume-job pairs have no meaningful textual overlap.
- The average similarity score of **0.0441** is relatively low because the majority of resumes are compared against unrelated job descriptions. Since every resume is compared with every job posting, most combinations naturally have very little similarity.
- This behavior is expected in a recommendation system and confirms that the model assigns higher scores only to relevant job descriptions while keeping unrelated matches close to zero.

In [133]:
# Best Matching Job

# Randomly select a resume
resume_index = random.randint(0, len(master_df) - 1)

# Find the job with the highest similarity score
best_match = similarity_matrix[resume_index].argmax()

# Display the selected resume title
print("Resume Title :", master_df.iloc[resume_index]["name"])

# Display the best matching job title
print("Best Matching Job :", job_df.iloc[best_match]["Job Title"])

# Display the similarity score
print("Similarity Score :", similarity_matrix[resume_index][best_match])

Resume Title : Database Administrator
Best Matching Job : Database Administrator
Similarity Score : 0.24898178999023973


### Observations

- The recommendation engine successfully retrieves job descriptions with high textual similarity.
- Similar resumes receive similar job recommendations.
- Multiple recommendations may have the same job title because different companies often post similar roles with different job descriptions.
- Higher cosine similarity scores indicate stronger textual similarity between a resume and a job description.

### Limitations
- TF-IDF focuses on keyword importance rather than semantic understanding.
- Synonyms and contextual meaning may not always be captured.
- The recommendation quality depends on the quality of the resume and job description text.
- More advanced embedding models such as Sentence Transformers or BERT can improve semantic matching in future versions.

### Model Serialization
**Overview:**
To prepare the AI Resume Screening System for deployment, the trained machine learning artifacts are serialized using **Joblib**. Serialization allows the application to load precomputed objects directly from disk instead of retraining or recomputing them every time the system starts.

**Artifacts Saved:**
- **TF-IDF Vectorizer** – Preserves the learned vocabulary and IDF values for transforming new resumes.
- **Cosine Similarity Matrix** – Stores precomputed similarity scores to enable fast job recommendations.
- **Processed Resume Dataset** – Maintains the processed resume information used during inference.
- **Processed Job Dataset** – Stores job metadata for mapping recommendation indices to job details.

**Benefits:**
- Faster application startup
- Reduced computation time
- Consistent predictions during inference
- Production-ready deployment
- Reusable machine learning pipeline

In [134]:
pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [136]:
# Import Joblib for saving Python objects
import joblib

# Save the trained TF-IDF vectorizer
joblib.dump(tfidf, "../artifacts/tfidf_vectorizer.joblib")

print("TF-IDF Vectorizer saved successfully.")

TF-IDF Vectorizer saved successfully.


In [137]:
# Save the cosine similarity matrix
joblib.dump(similarity_matrix, "../artifacts/similarity_matrix.joblib")

print("Similarity Matrix saved successfully.")

Similarity Matrix saved successfully.


In [138]:
# Save the processed resume dataset
joblib.dump(master_df, "../artifacts/processed_resume_dataset.joblib")

print("Processed Resume Dataset saved successfully.")

Processed Resume Dataset saved successfully.


In [139]:
# Save the processed job dataset
joblib.dump(job_df, "../artifacts/job_dataset.joblib")

print("Job Dataset saved successfully.")

Job Dataset saved successfully.


In [140]:
#Verify the saved artifacts
import os

print(os.listdir("../artifacts"))

['processed_resume_dataset.joblib', 'similarity_matrix.joblib', 'tfidf_vectorizer.joblib', 'job_dataset.joblib']
